In [1]:
import pandas as pd
import re
import time
import pickle
from deep_translator import GoogleTranslator
from transformers import pipeline

print("="*80)
print("FRESH TRANSLATION - Starting from scratch")
print("="*80)

# Step 1: Load and filter reviews
print("\n1. Loading reviews...")
df_reviews = pd.read_csv('olist_order_reviews_dataset.csv')

df_reviews_with_text = df_reviews[
    (df_reviews['review_comment_title'].notna()) | 
    (df_reviews['review_comment_message'].notna())
].copy()

print(f"   Reviews with text: {len(df_reviews_with_text)}")

FRESH TRANSLATION - Starting from scratch

1. Loading reviews...
   Reviews with text: 42706


In [2]:
# Step 2: Combine title and comment
print("\n2. Combining title and comment...")
df_reviews_with_text['review_text_combined'] = (
    df_reviews_with_text['review_comment_title'].fillna('') + ' ' + 
    df_reviews_with_text['review_comment_message'].fillna('')
).str.strip()


2. Combining title and comment...


In [3]:
# Step 3: Clean text
def clean_text(text):
    if pd.isna(text) or text == '':
        return ''
    text = re.sub(r'<[^>]+>', '', text)  # Remove HTML tags
    text = re.sub(r'\s+', ' ', text)      # Remove extra whitespace
    text = text.strip()
    return text

print("\n3. Cleaning text...")
df_reviews_with_text['review_text_combined'] = df_reviews_with_text['review_text_combined'].apply(clean_text)


3. Cleaning text...


In [4]:
# Step 4: Reset index to ensure clean iteration
df_reviews_with_text = df_reviews_with_text.reset_index(drop=True)
print(f"   Reset index. Total reviews: {len(df_reviews_with_text)}")

   Reset index. Total reviews: 42706


In [5]:
# Step 5: Translate with checkpoints
translator = GoogleTranslator(source='pt', target='en')


# Load sentiment analysis model (this will download ~500MB on first run)
print("   Loading sentiment analysis model (this may take a minute)...")
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=-1  # Use CPU; change to 0 for GPU if available
)
print("   ✅ Sentiment model loaded!")

checkpoint_file = 'translation_sentiment_checkpoint.pkl'

   Loading sentiment analysis model (this may take a minute)...


Device set to use cpu


   ✅ Sentiment model loaded!


In [6]:
# Check for existing checkpoint
try:
    with open(checkpoint_file, 'rb') as f:
        checkpoint_data = pickle.load(f)
        translated_texts = checkpoint_data['translations']
        sentiment_labels = checkpoint_data['sentiment_labels']
        sentiment_scores = checkpoint_data['sentiment_scores']
        start_idx = len(translated_texts)
    print(f"\n5. Found checkpoint - Resuming from review {start_idx}")
except:
    translated_texts = []
    sentiment_labels = []
    sentiment_scores = []
    start_idx = 0
    print(f"\n5. No checkpoint found - Starting fresh")

total_reviews = len(df_reviews_with_text)
print(f"   Processing {total_reviews - start_idx} reviews...")
print("="*80)



5. Found checkpoint - Resuming from review 5100
   Processing 37606 reviews...


In [7]:
# Process: Translate + Sentiment
for i in range(start_idx, total_reviews):
    text = df_reviews_with_text['review_text_combined'].iloc[i]
    
    try:
        if text == '':
            translated = ''
            sentiment_label = 'NEUTRAL'
            sentiment_score = 0.5
        else:
            # Truncate if too long
            if len(text) > 4500:
                text = text[:4500]
            
            # Translate
            translated = translator.translate(text)
            
            # Get sentiment (truncate to 512 tokens for model)
            sentiment_text = translated[:512] if len(translated) > 512 else translated
            sentiment_result = sentiment_analyzer(sentiment_text)[0]
            sentiment_label = sentiment_result['label']  # 'POSITIVE' or 'NEGATIVE'
            sentiment_score = sentiment_result['score']  # Confidence score
        
        # Store results
        translated_texts.append(translated)
        sentiment_labels.append(sentiment_label)
        sentiment_scores.append(sentiment_score)
        
        # Show progress every 10 reviews
        if len(translated_texts) % 10 == 0:
            percentage = (len(translated_texts) / total_reviews) * 100
            review_score = df_reviews_with_text['review_score'].iloc[i]
            print(f"\n[{len(translated_texts)}/{total_reviews}] {percentage:.1f}% complete")
            print(f"   Review Score: {review_score} stars")
            print(f"   PT: {text[:60]}...")
            print(f"   EN: {translated[:60]}...")
            print(f"   Sentiment: {sentiment_label} (confidence: {sentiment_score:.2f})")
            print("-" * 80)
        
        # Save checkpoint every 100 reviews
        if len(translated_texts) % 100 == 0:
            checkpoint_data = {
                'translations': translated_texts,
                'sentiment_labels': sentiment_labels,
                'sentiment_scores': sentiment_scores
            }
            with open(checkpoint_file, 'wb') as f:
                pickle.dump(checkpoint_data, f)
            print(f"\n{'='*80}")
            print(f"✅ CHECKPOINT SAVED: {len(translated_texts)}/{total_reviews} ({percentage:.1f}%)")
            print(f"{'='*80}\n")
        
        time.sleep(0.5)  # Rate limiting
        
    except Exception as e:
        print(f"\n❌ Error at review {i}: {e}")
        translated_texts.append(text)
        sentiment_labels.append('NEUTRAL')
        sentiment_scores.append(0.5)
        time.sleep(2)

print("\n" + "="*80)
print("✅ Translation + Sentiment Analysis complete!")
print("="*80)


[5110/42706] 12.0% complete
   Review Score: 5 stars
   PT: Super recomendo 16/07/201 Muito bom...
   EN: I highly recommend it 07/16/201 Very good...
   Sentiment: POSITIVE (confidence: 1.00)
--------------------------------------------------------------------------------

[5120/42706] 12.0% complete
   Review Score: 5 stars
   PT: ótimo produto, entrega rápida,muito satisfeito...
   EN: great product, fast delivery, very satisfied...
   Sentiment: POSITIVE (confidence: 1.00)
--------------------------------------------------------------------------------

[5130/42706] 12.0% complete
   Review Score: 3 stars
   PT: bom...
   EN: good...
   Sentiment: POSITIVE (confidence: 1.00)
--------------------------------------------------------------------------------

[5140/42706] 12.0% complete
   Review Score: 5 stars
   PT: 10...
   EN: 10...
   Sentiment: POSITIVE (confidence: 0.99)
--------------------------------------------------------------------------------

[5150/42706] 12.1% complet

In [8]:
# Step 6: Verify lengths
print(f"\nVerification:")
print(f"   Total reviews: {len(df_reviews_with_text)}")
print(f"   Total translations: {len(translated_texts)}")
print(f"   Total sentiment labels: {len(sentiment_labels)}")
print(f"   Total sentiment scores: {len(sentiment_scores)}")

if len(translated_texts) == len(sentiment_labels) == len(sentiment_scores) == len(df_reviews_with_text):
    print("   ✅ All lengths match!")
else:
    print("   ❌ Length mismatch!")
    raise ValueError("Lengths don't match!")



Verification:
   Total reviews: 42706
   Total translations: 42706
   Total sentiment labels: 42706
   Total sentiment scores: 42706
   ✅ All lengths match!


In [9]:
# Step 7: Add to dataframe
df_reviews_with_text['review_text_english'] = translated_texts
df_reviews_with_text['predicted_sentiment'] = sentiment_labels
df_reviews_with_text['sentiment_confidence'] = sentiment_score

In [10]:
# Step 8: Calculate sentiment-score alignment
def check_alignment(row):
    """Check if predicted sentiment matches review score"""
    score = row['review_score']
    sentiment = row['predicted_sentiment']
    
    # Expected sentiment based on score
    if score <= 2:
        expected = 'NEGATIVE'
    elif score >= 4:
        expected = 'POSITIVE'
    else:
        expected = 'NEUTRAL'  # Score 3
    
    # For binary sentiment (POSITIVE/NEGATIVE only)
    if sentiment == 'NEGATIVE' and score <= 2:
        return True
    elif sentiment == 'POSITIVE' and score >= 4:
        return True
    elif score == 3:
        return True  # Neutral, either sentiment is acceptable
    else:
        return False

df_reviews_with_text['is_aligned'] = df_reviews_with_text.apply(check_alignment, axis=1)

alignment_rate = df_reviews_with_text['is_aligned'].mean() * 100
print(f"\n   Sentiment-Score Alignment Rate: {alignment_rate:.1f}%")


   Sentiment-Score Alignment Rate: 90.9%


In [11]:
# Step 9: Parse dates
print("\n6. Parsing dates...")
df_reviews_with_text['review_creation_date'] = pd.to_datetime(
    df_reviews_with_text['review_creation_date']
)
df_reviews_with_text['review_answer_timestamp'] = pd.to_datetime(
    df_reviews_with_text['review_answer_timestamp']
)

# Step 10: Save final CSV
df_reviews_cleaned = df_reviews_with_text[[
    'review_id',
    'order_id',
    'review_score',
    'review_comment_title',
    'review_comment_message',
    'review_text_combined',
    'review_text_english',
    'predicted_sentiment',
    'sentiment_confidence',
    'is_aligned',
    'review_creation_date',
    'review_answer_timestamp'
]]

output_file = 'olist_order_reviews_cleaned.csv'
df_reviews_cleaned.to_csv(output_file, index=False)
print(f"\n✅ Saved {len(df_reviews_cleaned)} cleaned reviews to '{output_file}'")



6. Parsing dates...

✅ Saved 42706 cleaned reviews to 'olist_order_reviews_cleaned.csv'


In [12]:
print("\n" + "="*80)
print("Random sample verification:")
print("="*80)
import random
sample_indices = random.sample(range(len(df_reviews_cleaned)), 5)

for idx in sample_indices:
    row = df_reviews_cleaned.iloc[idx]
    print(f"\n--- Review {idx} ---")
    print(f"   Review Score: {row['review_score']} stars")
    print(f"   PT: {row['review_text_combined'][:80]}...")
    print(f"   EN: {row['review_text_english'][:80]}...")
    print(f"   Predicted: {row['predicted_sentiment']} (conf: {row['sentiment_confidence']:.2f})")
    print(f"   Aligned: {'✅' if row['is_aligned'] else '❌'}")


Random sample verification:

--- Review 16656 ---
   Review Score: 5 stars
   PT: Produto de qualidade...
   EN: Quality product...
   Predicted: POSITIVE (conf: 1.00)
   Aligned: ✅

--- Review 20887 ---
   Review Score: 5 stars
   PT: Entra antes do prazo estabelecido....
   EN: Enter before the established deadline....
   Predicted: NEGATIVE (conf: 1.00)
   Aligned: ❌

--- Review 13762 ---
   Review Score: 1 stars
   PT: Eu comprei um produto e me enviaram outro, comprei capa e película para o redmi ...
   EN: I bought a product and they sent me another, I bought a cover and film for the R...
   Predicted: NEGATIVE (conf: 1.00)
   Aligned: ✅

--- Review 33093 ---
   Review Score: 5 stars
   PT: Produto entregue....
   EN: Product delivered....
   Predicted: POSITIVE (conf: 1.00)
   Aligned: ✅

--- Review 4194 ---
   Review Score: 1 stars
   PT: Comprei para presente de Natal dia 27/11, ou seja quase um mês antes, a promessa...
   EN: I bought it for a Christmas present on 11/27, tha

In [13]:
print("\n" + "="*80)
print("ALL DONE! ✅")
print("="*80)
print("\nFinal CSV includes:")
print("   - Original Portuguese text")
print("   - English translation")
print("   - Predicted sentiment (POSITIVE/NEGATIVE)")
print("   - Sentiment confidence score (0-1)")
print("   - Alignment flag (does sentiment match review score?)")


ALL DONE! ✅

Final CSV includes:
   - Original Portuguese text
   - English translation
   - Predicted sentiment (POSITIVE/NEGATIVE)
   - Sentiment confidence score (0-1)
   - Alignment flag (does sentiment match review score?)
